# Code to run the ml_pipeline project

### Config Loader

In [14]:
from ml_pipeline.config_loader import load_config

config = load_config("configs/example_run.json")
print(config["run_name"])
print(config["data"]["target_column"])
print(config["model"]["type"])
print(config["model"]["task"])
print(config["model"]["hyperparameters"].get("n_estimators"))

iris_baseline_v1
X
xgboost
classification
100


In [1]:
from ml_pipeline.config_loader import load_config
from ml_pipeline.ingest import load_data, prepare_data, split_data

config = load_config("configs/example_run.json")

df = load_data(config["data"]["raw_data_path"])
X, y = prepare_data(df, config["data"]["target_column"], config["data"]["exclude_columns"])
X_train, X_test, y_train, y_test = split_data(X, y, config["data"]["test_size"], config["data"]["random_state"])

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(120, 4) (30, 4)
X
1    41
0    40
2    39
Name: count, dtype: int64


### Train the Model

In [2]:
from ml_pipeline.train import train_model

model = train_model(
    X_train, y_train,
    config["model"]["type"],
    config["model"]["task"],
    config["model"]["hyperparameters"]
)

print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)


### Evaluate Model

In [3]:
from ml_pipeline.evaluate import evaluate_model

results = evaluate_model(
    model, X_test, y_test,
    config["evaluation"]["metrics"]
)

print(results)

{'accuracy': 1.0, 'f1': 1.0}


### Run whole pipeline with MLFlow tracking

In [2]:
# Import the run_pipeline function from pipeline.py
from ml_pipeline.pipeline import run_pipeline

# run_pipeline only has one argument, the path to the config file
model, results = run_pipeline(r"C:\Python\ml_pipeline\configs\example_run.json")

Starting run: iris_baseline_v1
Tracking URI: file:///C:/Python/ml_pipeline/mlruns
Experiment ID: 851726502844455103
Experiment name: Iris_Experiment
Data loaded: 120 train rows, 30 test rows
Model trained: xgboost
Evaluation results: {'accuracy': 1.0, 'f1': 1.0}


2026/04/28 07:02:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model saved to outputs\iris_baseline_v1\model.joblib
Results saved to outputs\iris_baseline_v1\results.json
